# Modèles d'Ensemble — Random Forest, XGBoost, LightGBM

Ce notebook entraîne et évalue trois modèles d'ensemble sur le jeu de données
Kaggle Credit Card Fraud Detection. Les méthodes d'ensemble (bagging, boosting)
sont parmi les plus performantes sur les données tabulaires.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.ensemble_models import create_random_forest, create_xgboost, create_lightgbm
from src.evaluation.metrics import compute_all_metrics, compute_classification_report
from src.evaluation.visualizer import plot_confusion_matrix, plot_roc_curves, plot_pr_curves

plt.rcParams.update({'font.size': 12, 'figure.dpi': 300, 'font.family': 'serif'})
sns.set_style('whitegrid')

FIGURES_DIR = os.path.join('..', 'reports', 'figures', 'models')
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Imports chargés avec succès.")

In [ ]:
# ── Chargement et préparation des données ──
from src.data.loader import load_creditcard
from src.data.preprocessor import FraudPreprocessor
from src.data.splitter import stratified_split

# Charger le dataset
df = load_creditcard()

# Prétraitement
preprocessor = FraudPreprocessor(scaler_type='standard')
df = preprocessor.handle_missing(df)
df = preprocessor.engineer_features_kaggle(df)

# Séparer features et cible
TARGET = 'Class'
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES]
y = df[TARGET]

# Normalisation
X_scaled = preprocessor.fit_transform(X)

# Split stratifié train / val / test
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(X_scaled, y)

# Calculer scale_pos_weight pour XGBoost
n_legit = (y_train == 0).sum()
n_fraud = (y_train == 1).sum()
scale_pos_weight = n_legit / n_fraud

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Scale pos weight (XGBoost): {scale_pos_weight:.1f}")

## Random Forest

Le Random Forest combine plusieurs arbres de décision par *bagging*, réduisant ainsi
le sur-apprentissage inhérent à un arbre individuel. Nous utilisons 200 estimateurs.

In [ ]:
# ── Random Forest : entraînement et évaluation ──
import time

rf_model = create_random_forest(n_estimators=200)

start = time.time()
rf_model.fit(X_train, y_train)
rf_time = time.time() - start

# Prédictions
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Métriques
metrics_rf = compute_all_metrics(y_test, y_pred_rf, y_proba_rf)
print(f"=== Random Forest — Résultats (entraîné en {rf_time:.1f}s) ===\n")
print(compute_classification_report(y_test, y_pred_rf))
print(f"AUC-ROC: {metrics_rf['auc_roc']:.4f}")
print(f"AUPRC:   {metrics_rf['auprc']:.4f}")
print(f"F1:      {metrics_rf['f1_score']:.4f}")
print(f"Rappel:  {metrics_rf['recall']:.4f}")

In [ ]:
# ── Matrice de confusion + Feature Importance — Random Forest ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrix (left)
from sklearn.metrics import confusion_matrix as sk_cm
cm_rf = sk_cm(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Légitime', 'Fraude'],
            yticklabels=['Légitime', 'Fraude'], ax=axes[0])
axes[0].set_xlabel('Prédiction')
axes[0].set_ylabel('Réalité')
axes[0].set_title('Matrice de Confusion — Random Forest')

# Feature importance (right) — top 15
feature_names = X_train.columns if hasattr(X_train, 'columns') else [f'F{i}' for i in range(X_train.shape[1])]
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

feat_imp.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_xlabel('Importance')
axes[1].set_title('Top 15 Features — Random Forest')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'rf_cm_feature_importance.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## XGBoost

XGBoost utilise le *gradient boosting* avec régularisation. Le paramètre `scale_pos_weight`
compense le déséquilibre des classes. Nous utilisons 300 estimateurs.

In [ ]:
# ── XGBoost : entraînement et évaluation ──
xgb_model = create_xgboost(scale_pos_weight=scale_pos_weight, n_estimators=300)

start = time.time()
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - start

# Prédictions
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Métriques
metrics_xgb = compute_all_metrics(y_test, y_pred_xgb, y_proba_xgb)
print(f"=== XGBoost — Résultats (entraîné en {xgb_time:.1f}s) ===\n")
print(compute_classification_report(y_test, y_pred_xgb))
print(f"AUC-ROC: {metrics_xgb['auc_roc']:.4f}")
print(f"AUPRC:   {metrics_xgb['auprc']:.4f}")
print(f"F1:      {metrics_xgb['f1_score']:.4f}")
print(f"Rappel:  {metrics_xgb['recall']:.4f}")

In [ ]:
# ── Matrice de confusion + Feature Importance — XGBoost ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrix (left)
cm_xgb = sk_cm(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Légitime', 'Fraude'],
            yticklabels=['Légitime', 'Fraude'], ax=axes[0])
axes[0].set_xlabel('Prédiction')
axes[0].set_ylabel('Réalité')
axes[0].set_title('Matrice de Confusion — XGBoost')

# Feature importance (right) — top 15
importances_xgb = xgb_model.feature_importances_
feat_imp_xgb = pd.Series(importances_xgb, index=feature_names).sort_values(ascending=False).head(15)

feat_imp_xgb.plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='black')
axes[1].set_xlabel('Importance')
axes[1].set_title('Top 15 Features — XGBoost')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'xgb_cm_feature_importance.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## LightGBM

LightGBM utilise un algorithme de *gradient boosting* optimisé avec croissance par feuille
(*leaf-wise*), ce qui le rend particulièrement rapide. Le paramètre `is_unbalance=True`
gère le déséquilibre des classes.

In [ ]:
# ── LightGBM : entraînement et évaluation ──
lgbm_model = create_lightgbm(n_estimators=300)

start = time.time()
lgbm_model.fit(X_train, y_train)
lgbm_time = time.time() - start

# Prédictions
y_pred_lgbm = lgbm_model.predict(X_test)
y_proba_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

# Métriques
metrics_lgbm = compute_all_metrics(y_test, y_pred_lgbm, y_proba_lgbm)
print(f"=== LightGBM — Résultats (entraîné en {lgbm_time:.1f}s) ===\n")
print(compute_classification_report(y_test, y_pred_lgbm))
print(f"AUC-ROC: {metrics_lgbm['auc_roc']:.4f}")
print(f"AUPRC:   {metrics_lgbm['auprc']:.4f}")
print(f"F1:      {metrics_lgbm['f1_score']:.4f}")
print(f"Rappel:  {metrics_lgbm['recall']:.4f}")

In [ ]:
# ── Matrice de confusion — LightGBM ──
plot_confusion_matrix(y_test, y_pred_lgbm, model_name="LightGBM",
                      save_path="models/cm_lightgbm")
plt.show()

## Comparaison des Ensembles

Courbes ROC, courbes Précision-Rappel et tableau récapitulatif des trois modèles d'ensemble.

In [ ]:
# ── Courbes ROC — Modèles d'Ensemble ──
roc_models = {
    "Random Forest": (y_test, y_proba_rf),
    "XGBoost":       (y_test, y_proba_xgb),
    "LightGBM":      (y_test, y_proba_lgbm),
}

plot_roc_curves(roc_models, save_path="models/roc_curves_ensembles")
plt.show()

print("Figure sauvegardée dans reports/figures/models/roc_curves_ensembles.png")

In [ ]:
# ── Courbes Précision-Rappel — Modèles d'Ensemble ──
pr_models = {
    "Random Forest": (y_test, y_proba_rf),
    "XGBoost":       (y_test, y_proba_xgb),
    "LightGBM":      (y_test, y_proba_lgbm),
}

plot_pr_curves(pr_models, save_path="models/pr_curves_ensembles")
plt.show()

print("Figure sauvegardée dans reports/figures/models/pr_curves_ensembles.png")

In [ ]:
# ── Tableau comparatif des modèles d'ensemble ──
comparison_data = {
    "Modèle": ["Random Forest", "XGBoost", "LightGBM"],
    "AUC-ROC": [metrics_rf['auc_roc'], metrics_xgb['auc_roc'], metrics_lgbm['auc_roc']],
    "AUPRC":   [metrics_rf['auprc'], metrics_xgb['auprc'], metrics_lgbm['auprc']],
    "F1-Score": [metrics_rf['f1_score'], metrics_xgb['f1_score'], metrics_lgbm['f1_score']],
    "Précision": [metrics_rf['precision'], metrics_xgb['precision'], metrics_lgbm['precision']],
    "Rappel":   [metrics_rf['recall'], metrics_xgb['recall'], metrics_lgbm['recall']],
    "Spécificité": [metrics_rf['specificity'], metrics_xgb['specificity'], metrics_lgbm['specificity']],
    "Temps (s)": [round(rf_time, 1), round(xgb_time, 1), round(lgbm_time, 1)],
}

df_comp = pd.DataFrame(comparison_data).set_index("Modèle")
display(df_comp.style.format({
    "AUC-ROC": "{:.4f}", "AUPRC": "{:.4f}", "F1-Score": "{:.4f}",
    "Précision": "{:.4f}", "Rappel": "{:.4f}", "Spécificité": "{:.4f}",
    "Temps (s)": "{:.1f}"
}).highlight_max(axis=0, subset=["AUC-ROC", "AUPRC", "F1-Score", "Rappel"], color='lightgreen')
 .highlight_min(axis=0, subset=["Temps (s)"], color='lightyellow'))

## Observations

1. **XGBoost — meilleur individuel** : XGBoost obtient généralement le meilleur AUC-ROC et
   F1-score parmi les trois modèles. Son mécanisme de *gradient boosting* avec régularisation
   L1/L2 s'adapte particulièrement bien aux données tabulaires déséquilibrées.

2. **LightGBM — le plus rapide** : Grâce à son algorithme de croissance par feuille et
   l'échantillonnage par histogramme (GOSS), LightGBM s'entraîne significativement plus vite
   que les deux autres, tout en maintenant des performances très compétitives. Idéal pour la
   mise en production.

3. **Random Forest — le plus stable** : Le Random Forest, par construction (bagging +
   sous-échantillonnage de features), présente la variance la plus faible entre différents
   *runs*. Son interprétabilité via l'importance des features en fait un bon candidat pour
   l'analyse exploratoire.

4. **Features discriminantes** : Les trois modèles convergent sur les mêmes features
   importantes (V14, V4, V12, V17), ce qui confirme les observations de l'EDA.

5. **Prochaine étape** : Ces modèles d'ensemble constitueront la base de notre *stacking*
   final. Le meilleur modèle individuel sera également comparé aux approches de deep learning.